# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO

df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total Revenue: $" + str(round(total_revenue, 2)))
print("Across all 400 orders, we generated $" + str(round(total_revenue, 2)) + " in total revenue.")
print("")
print("Total Units Sold: " + str(total_units))
print("We moved " + str(total_units) + " units across all vendors and categories.")

# Created revenue column (qty * price), calculated total revenue and total units

Total Revenue: $8520.0
Across all 400 orders, we generated $8520.0 in total revenue.

Total Units Sold: 783
We moved 783 units across all vendors and categories.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO

by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
by_category_pct = (by_category / by_category.sum()) * 100

q2_result = pd.DataFrame({
    'Revenue': by_category,
    'Percentage': by_category_pct.round(2)
})

print("Revenue by Category (Highest to Lowest)")
print("")
print(q2_result)
print("Food accounts for " + str(round(q2_result.loc['Food', 'Percentage'], 1)) + "% of revenue.")
print("RainGear is the smallest segment at " + str(round(q2_result.loc['RainGear', 'Percentage'], 1)) + "%.")

# Grouped revenue by category, sorted highest to lowest, calculated percentage share

Revenue by Category (Highest to Lowest)

          Revenue  Percentage
category                     
Food       4293.0       50.39
Merch      1771.5       20.79
Drink      1554.0       18.24
RainGear    901.5       10.58
Food accounts for 50.4% of revenue.
RainGear is the smallest segment at 10.6%.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO

q3_result = df.groupby('vendor_id').agg({
    'revenue': ['mean', 'count']
}).round(2)

q3_result.columns = ['Avg Revenue', 'Order Count']
q3_result = q3_result.sort_values('Avg Revenue', ascending=False)

print("")
print("Average Order Revenue by Vendor")
print("")
print(q3_result)
print("")
highest_vendor = q3_result.index[0]
highest_avg = q3_result.iloc[0, 0]
highest_count = int(q3_result.iloc[0, 1])
print(highest_vendor + " has the highest average order revenue at $" + str(highest_avg) + ".")
print("This is based on " + str(highest_count) + " orders.")

# Grouped by vendor, calculated mean revenue and order count, sorted by highest average revenue


Average Order Revenue by Vendor

           Avg Revenue  Order Count
vendor_id                          
V-01             22.60           94
V-18             21.75          108
V-05             20.58           93
V-10             20.31          105

V-01 has the highest average order revenue at $22.6.
This is based on 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO

merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_pct = (merch_revenue / total_revenue) * 100

print("Merch Revenue as Percentage")
print("")
print("Merch Revenue Percentage: " + str(round(merch_pct, 1)) + "%")
print("Merch accounts for " + str(round(merch_pct, 1)) + "% of total revenue.")
print("Dollar amount: $" + str(round(merch_revenue, 2)))

# Filtered for Merch category, calculated revenue amount and percentage of total

Merch Revenue as Percentage

Merch Revenue Percentage: 20.8%
Merch accounts for 20.8% of total revenue.
Dollar amount: $1771.5


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

#find unmatched vendors
unmatched = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()
print("Unmatched vendors found: " + str(list(unmatched)))

# Perform the merge
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Validate results
print("")
print("Validation Results:")
print("Original row count: " + str(len(df)))
print("Joined row count: " + str(len(joined)))
print("Row counts match: " + str(len(df) == len(joined)))
print("")
print("Original revenue total: $" + str(round(total_revenue, 2)))
print("Joined revenue total: $" + str(round(joined['revenue'].sum(), 2)))

revenue_match = abs(total_revenue - joined['revenue'].sum()) < 0.01

print("Revenue totals match: " + str(revenue_match))
print("")
print("UNMATCHED VENDOR: " + unmatched[0])
print("This vendor is in the data but NOT in the lookup table.")
print("Action taken: Left join preserved all rows. " + unmatched[0] + " now has vendor_name = NaN.")
print("Recommendation: Add " + unmatched[0] + " to the lookup or investigate if this is a data error.")

# Created vendor lookup table, performed LEFT JOIN merge, found unmatched vendor V-18, validated row/revenue counts

Unmatched vendors found: ['V-18']

Validation Results:
Original row count: 400
Joined row count: 400
Row counts match: True

Original revenue total: $8520.0
Joined revenue total: $8520.0
Revenue totals match: True

UNMATCHED VENDOR: V-18
This vendor is in the data but NOT in the lookup table.
Action taken: Left join preserved all rows. V-18 now has vendor_name = NaN.
Recommendation: Add V-18 to the lookup or investigate if this is a data error.


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO

pivot = joined.pivot_table(
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='TOTAL'
).round(2)

print("Pivot Table - Revenue by Vendor and Category")
print(pivot)
print("")
print("This matrix shows revenue generated by each vendor across product categories.")
print("The TOTAL row shows revenue by category.")
print("The TOTAL column shows total revenue by vendor.")

# Created pivot table with vendors as rows, categories as columns, revenue values, added row/column totals

Pivot Table - Revenue by Vendor and Category
category         Drink    Food   Merch  RainGear   TOTAL
vendor_name                                             
Cav Merch North  502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers     171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos    298.5   882.0   489.0     244.5  1914.0
TOTAL            972.0  3274.5  1263.0     661.5  6171.0

This matrix shows revenue generated by each vendor across product categories.
The TOTAL row shows revenue by category.
The TOTAL column shows total revenue by vendor.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category.sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** Food accounts for 50.4% of our 8,520 dollars in revenue while RainGear accounts for only 10.6% (901.50 dollars). This suggests that we should either invest more in RainGear marketing and inventory or reconsider if raingear fits with the rest of our product offerings. Additionally, V-01 vendors generate 22.60 dollars on average per order compared to other vendors at 0.31 dollars. This indicates that V-01 vendors have a better pricing or marketing strategy.


**b)** Question 6 is the least trustworthy because vendor V-18 is unmatched in the vendor lookup table. V-18's 108 orders (27% of all orders) and 2,349 dollars in revenue are excluded from the vendor analysis. The pivot table shows only 6,171 dollars of total revenue across three vendors instead of the correct 8,520 dollars. While the LEFT JOIN correctly preserved all rows from the main data, the pivot table does not give us a complete view of vendor performance because vendor V-18 data is excluded from the report.